# 📈 Project 05 — End-to-End Data Analysis (NumPy + Pandas)

**Scenario:** you are the analyst for a small online store. The owner asks: *Which products and regions make money? Is revenue growing? Who are the best customers? Where are we losing sales to discounts and returns?*

**Skills used:** generating realistic data with NumPy, cleaning, feature engineering, `groupby`, pivot tables, time-series resampling, customer segmentation (RFM), outlier detection, correlation, and a written summary. Charts are shown as optional Matplotlib cells.

**Workflow (the standard data-science loop):** Ask → Load → Clean → Explore → Analyse → Communicate.

> ℹ️ The data below is **simulated** (random numbers with a fixed seed), so the "findings" are illustrations of the method, not facts about any real market. For instance, discounts show no effect on quantity simply because the generator made them independent. Re-run the same steps on a real dataset for real conclusions.

---
## 1. Create a realistic dataset
(Real data would come from `pd.read_csv("sales.csv")`. We simulate one, with a fixed seed so results are reproducible.)

In [1]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(2025)
N = 4000

products = pd.DataFrame({
    "product": ["Laptop", "Phone", "Headphones", "Keyboard", "Mouse", "Monitor", "Webcam", "Charger"],
    "category": ["Computers", "Mobile", "Audio", "Accessories", "Accessories", "Computers", "Accessories", "Mobile"],
    "price":   [55000, 30000, 4500, 1800, 900, 14000, 3200, 1200],
})
regions = ["North", "South", "East", "West"]

sales = pd.DataFrame({
    "order_id": np.arange(1, N + 1),
    "date": pd.to_datetime("2024-01-01") + pd.to_timedelta(rng.integers(0, 365, N), unit="D"),
    "customer_id": rng.integers(1, 600, N),
    "product": rng.choice(products["product"], N, p=[.05, .10, .18, .17, .20, .07, .08, .15]),
    "region": rng.choice(regions, N, p=[.30, .25, .20, .25]),
    "quantity": rng.integers(1, 5, N),
    "discount": rng.choice([0, 0.05, 0.10, 0.20], N, p=[.55, .25, .15, .05]),
    "returned": rng.random(N) < 0.04,
})
# Inject realistic mess: missing values, duplicates, a data-entry error
sales.loc[rng.choice(N, 40, replace=False), "region"] = None
sales.loc[rng.choice(N, 25, replace=False), "quantity"] = np.nan
sales = pd.concat([sales, sales.sample(15, random_state=1)], ignore_index=True)     # 15 duplicate rows
sales.loc[3, "quantity"] = 400                                                        # typo: 400 instead of 4

print(sales.shape)
sales.head()

(4015, 8)


   order_id       date  customer_id  ... quantity discount  returned
0         1 2024-06-12          377  ...      4.0      0.0     False
1         2 2024-12-28          425  ...      1.0      0.0     False
2         3 2024-12-28          283  ...      1.0      0.0     False
3         4 2024-05-19          292  ...    400.0      0.0     False
4         5 2024-12-14          365  ...      3.0      0.0     False

[5 rows x 8 columns]

## 2. Inspect and clean

In [2]:
print(sales.dtypes, "\n")
print("missing values:\n", sales.isna().sum()[lambda s: s > 0], "\n")
print("duplicate rows:", sales.duplicated().sum())
print(sales["quantity"].describe().round(2))

order_id                int64
date           datetime64[us]
customer_id             int64
product                   str
region                    str
quantity              float64
discount              float64
returned                 bool
dtype: object 

missing values:
 region      40
quantity    25
dtype: int64 

duplicate rows: 15
count    3990.00
mean        2.61
std         6.39
min         1.00
25%         2.00
50%         3.00
75%         3.00
max       400.00
Name: quantity, dtype: float64


In [3]:
clean = (sales
         .drop_duplicates()
         .assign(region=lambda d: d["region"].fillna("Unknown"))
         .dropna(subset=["quantity"])
         .query("quantity <= 10")                # remove the impossible 400
         .astype({"quantity": int})
         .merge(products, on="product", how="left"))

# Feature engineering: revenue, month, weekday, net revenue (returns don't count)
clean["gross"] = clean["price"] * clean["quantity"]
clean["revenue"] = clean["gross"] * (1 - clean["discount"])
clean["net_revenue"] = np.where(clean["returned"], 0, clean["revenue"])
clean["month"] = clean["date"].dt.to_period("M")
clean["weekday"] = clean["date"].dt.day_name()

print(f"rows: {len(sales)} -> {len(clean)}  (removed {len(sales) - len(clean)} bad rows)")
clean.head(3)

rows: 4015 -> 3974  (removed 41 bad rows)


   order_id       date  customer_id  ... net_revenue    month    weekday
0         1 2024-06-12          377  ...      7200.0  2024-06  Wednesday
1         2 2024-12-28          425  ...      1800.0  2024-12   Saturday
2         3 2024-12-28          283  ...      4500.0  2024-12   Saturday

[3 rows x 15 columns]

## 3. Explore: the headline numbers

In [4]:
total = clean["net_revenue"].sum()
print(f"Net revenue       : ₹{total:,.0f}")
print(f"Orders            : {len(clean):,}")
print(f"Average order     : ₹{clean['net_revenue'].mean():,.0f}   (median ₹{clean['net_revenue'].median():,.0f} -> skewed by big-ticket items)")
print(f"Return rate       : {clean['returned'].mean():.1%}")
print(f"Revenue lost to returns   : ₹{clean.loc[clean['returned'], 'revenue'].sum():,.0f}")
print(f"Revenue given as discounts: ₹{(clean['gross'] - clean['revenue']).sum():,.0f}")

Net revenue       : ₹76,652,335
Orders            : 3,974
Average order     : ₹19,288   (median ₹4,560 -> skewed by big-ticket items)
Return rate       : 4.1%
Revenue lost to returns   : ₹3,148,400
Revenue given as discounts: ₹2,956,365


## 4. Which products and categories earn the most?

In [5]:
by_product = (clean.groupby("product")
              .agg(orders=("order_id", "count"), units=("quantity", "sum"), net=("net_revenue", "sum"))
              .assign(share=lambda d: (d["net"] / d["net"].sum() * 100).round(1))
              .sort_values("net", ascending=False))
print(by_product.round(0).to_string())

print("\nBy category:")
print(clean.groupby("category")["net_revenue"].sum().sort_values(ascending=False).apply(lambda v: f"₹{v:,.0f}").to_string())

top2_share = by_product["share"].head(2).sum()
print(f"\n👉 Insight: top-2 products generate {top2_share:.0f}% of revenue although they are only 2 of 8 items sold.")

            orders  units         net  share
product                                     
Phone          389   1005  27279000.0   36.0
Laptop         172    460  24219250.0   32.0
Monitor        294    700   8995000.0   12.0
Headphones     725   1835   7617600.0   10.0
Keyboard       711   1745   2924190.0    4.0
Webcam         321    813   2377280.0    3.0
Mouse          776   1974   1649115.0    2.0
Charger        586   1442   1590900.0    2.0

By category:
category
Computers      ₹33,214,250
Mobile         ₹28,869,900
Audio           ₹7,617,600
Accessories     ₹6,950,585

👉 Insight: top-2 products generate 67% of revenue although they are only 2 of 8 items sold.


## 5. Regions and a pivot table

In [6]:
pivot = clean.pivot_table(values="net_revenue", index="region", columns="category", aggfunc="sum", fill_value=0, margins=True)
print((pivot / 1e5).round(1).to_string())         # in lakhs
print("\n(₹ lakhs)  → 'All' row/column are totals")

category  Accessories  Audio  Computers  Mobile    All
region                                                
East             12.7   14.3       68.0    66.3  161.3
North            21.4   22.9      100.1    80.1  224.6
South            19.0   20.8       72.1    59.6  171.5
Unknown           0.7    0.7        1.6     0.8    3.9
West             15.6   17.4       90.3    81.9  205.3
All              69.5   76.2      332.1   288.7  766.5

(₹ lakhs)  → 'All' row/column are totals


## 6. Trend: monthly revenue and growth

In [7]:
monthly = clean.groupby("month")["net_revenue"].sum()
growth = monthly.pct_change().mul(100).round(1)
trend = pd.DataFrame({"revenue_lakh": (monthly / 1e5).round(1), "mom_growth_%": growth})
print(trend.to_string())

best, worst = monthly.idxmax(), monthly.idxmin()
print(f"\nBest month: {best}   Worst month: {worst}")

# 3-month rolling average smooths noise
rolling = monthly.rolling(3).mean()
print("\nRolling 3-month average (lakh):", (rolling / 1e5).round(1).dropna().tolist())

# Day-of-week pattern
dow = clean.groupby("weekday")["net_revenue"].sum().reindex(["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"])
print("\nBy weekday (lakh):", (dow / 1e5).round(1).to_dict())

         revenue_lakh  mom_growth_%
month                              
2024-01          66.6           NaN
2024-02          57.5         -13.7
2024-03          64.4          12.1
2024-04          64.6           0.2
2024-05          70.5           9.1
2024-06          70.3          -0.2
2024-07          69.2          -1.6
2024-08          65.6          -5.3
2024-09          62.3          -5.0
2024-10          57.9          -7.1
2024-11          63.8          10.3
2024-12          53.8         -15.6

Best month: 2024-05   Worst month: 2024-12

Rolling 3-month average (lakh): [62.9, 62.2, 66.5, 68.5, 70.0, 68.4, 65.7, 61.9, 61.3, 58.5]

By weekday (lakh): {'Monday': 109.7, 'Tuesday': 105.6, 'Wednesday': 101.6, 'Thursday': 120.1, 'Friday': 92.4, 'Saturday': 102.8, 'Sunday': 134.4}


## 7. Do discounts help? (correlation & comparison)

In [8]:
by_disc = clean.groupby("discount").agg(orders=("order_id", "count"), avg_qty=("quantity", "mean"), avg_net=("net_revenue", "mean")).round(2)
print(by_disc.to_string())
print("\ncorrelation(discount, quantity) =", round(clean["discount"].corr(clean["quantity"]), 3))
print("→ near 0: in this data discounts do NOT increase units per order; they just cost margin.")
print("\nCorrelation matrix:\n", clean[["price", "quantity", "discount", "net_revenue"]].corr().round(2).to_string())

          orders  avg_qty   avg_net
discount                           
0.00        2211     2.53  19938.85
0.05         984     2.49  19757.20
0.10         607     2.48  17437.31
0.20         172     2.42  14779.07

correlation(discount, quantity) = -0.026
→ near 0: in this data discounts do NOT increase units per order; they just cost margin.

Correlation matrix:
              price  quantity  discount  net_revenue
price         1.00      0.03     -0.01         0.88
quantity      0.03      1.00     -0.03         0.25
discount     -0.01     -0.03      1.00        -0.03
net_revenue   0.88      0.25     -0.03         1.00


## 8. Customer segmentation with RFM
**R**ecency (days since last order), **F**requency (orders), **M**onetary (spend) — the classic marketing model.

In [9]:
snapshot = clean["date"].max() + pd.Timedelta(days=1)
rfm = (clean.groupby("customer_id")
       .agg(recency=("date", lambda d: (snapshot - d.max()).days),
            frequency=("order_id", "count"),
            monetary=("net_revenue", "sum")))
# Score each metric 1-4 using quartiles (recency: lower is better)
rfm["R"] = pd.qcut(rfm["recency"], 4, labels=[4, 3, 2, 1]).astype(int)
rfm["F"] = pd.qcut(rfm["frequency"].rank(method="first"), 4, labels=[1, 2, 3, 4]).astype(int)
rfm["M"] = pd.qcut(rfm["monetary"], 4, labels=[1, 2, 3, 4]).astype(int)
rfm["score"] = rfm["R"] + rfm["F"] + rfm["M"]
rfm["segment"] = pd.cut(rfm["score"], [0, 5, 8, 10, 12], labels=["At risk", "Regular", "Loyal", "Champion"])

print(rfm["segment"].value_counts().to_string())
print("\nRevenue by segment:")
print((rfm.groupby("segment", observed=True)["monetary"].sum() / rfm["monetary"].sum() * 100).round(1).to_string())
print("\nTop 5 customers:\n", rfm.sort_values("monetary", ascending=False).head(5)[["frequency", "monetary", "segment"]].round(0).to_string())

segment
Regular     222
Loyal       171
At risk     140
Champion     65

Revenue by segment:
segment
At risk      6.8
Regular     31.0
Loyal       41.3
Champion    20.9

Top 5 customers:
              frequency  monetary   segment
customer_id                               
483                 10  582900.0     Loyal
507                 11  537995.0     Loyal
161                  9  513900.0     Loyal
362                 11  489575.0     Loyal
466                 10  469620.0  Champion


## 9. Outliers (IQR rule) and data-quality checks

In [10]:
q1, q3 = clean["net_revenue"].quantile([.25, .75])
iqr = q3 - q1
upper = q3 + 1.5 * iqr
outliers = clean[clean["net_revenue"] > upper]
print(f"IQR fence: ₹{upper:,.0f}; {len(outliers)} orders above it ({len(outliers)/len(clean):.1%}) → mostly Laptops/Monitors/Phones:")
print(outliers["product"].value_counts().head(4).to_string())

# sanity checks (assertions document your assumptions)
assert clean["order_id"].is_unique
assert (clean["quantity"] > 0).all()
assert clean["discount"].between(0, 1).all()
print("\nAll data-quality assertions passed ✔")

IQR fence: ₹30,139; 584 orders above it (14.7%) → mostly Laptops/Monitors/Phones:
product
Phone      287
Laptop     169
Monitor    128

All data-quality assertions passed ✔


## 10. Optional charts (Matplotlib)

In [ ]:
# notest
try:
    import matplotlib.pyplot as plt
except ImportError:
    print('Matplotlib is not installed. Run:  pip install matplotlib   then re-run this cell.')
else:
    fig, ax = plt.subplots(2, 2, figsize=(12, 8))
    by_product["net"].sort_values().plot.barh(ax=ax[0, 0], title="Net revenue by product")
    (monthly / 1e5).plot(ax=ax[0, 1], marker="o", title="Monthly revenue (lakh)")
    pivot.drop("All").drop(columns="All").div(1e5).plot.bar(stacked=True, ax=ax[1, 0], title="Region × category (lakh)")
    clean["net_revenue"].plot.hist(bins=40, ax=ax[1, 1], title="Order value distribution")
    plt.tight_layout(); plt.show()

## 11. Communicate: write the findings
Good analysis ends with **plain-language recommendations**, not code. Generate a summary directly from the data so it never goes stale:

In [11]:
top_prod = by_product.index[0]
best_region = clean.groupby("region")["net_revenue"].sum().drop("Unknown", errors="ignore").idxmax()
champions = int((rfm["segment"] == "Champion").sum())
champ_share = rfm.loc[rfm["segment"] == "Champion", "monetary"].sum() / rfm["monetary"].sum()

report = f"""
SALES REPORT — 2024
===================
• Net revenue ₹{total/1e7:,.2f} crore from {len(clean):,} orders (avg ₹{clean['net_revenue'].mean():,.0f}).
• {top_prod} is the #1 product ({by_product['share'].iloc[0]}% of revenue); the top-2 products give {top2_share:.0f}%.
• {best_region} is the strongest region.
• Best month: {best}; weakest: {worst}.
• {champions} 'Champion' customers ({champ_share:.0%} of revenue) — protect them with loyalty offers.
• Returns cost ₹{clean.loc[clean['returned'], 'revenue'].sum()/1e5:,.1f} lakh ({clean['returned'].mean():.1%} of orders).
• Discounts cost ₹{(clean['gross'] - clean['revenue']).sum()/1e5:,.1f} lakh without lifting units per order → review the discount policy.

RECOMMENDATIONS
1. Keep the top products in stock; bundle low-margin accessories with them.
2. Target 'At risk' customers with a win-back campaign.
3. Cap discounts at 10 % and test the effect (A/B test).
4. Investigate the causes of returns for the highest-value items.
"""
print(report)


SALES REPORT — 2024
• Net revenue ₹7.67 crore from 3,974 orders (avg ₹19,288).
• Phone is the #1 product (35.6% of revenue); the top-2 products give 67%.
• North is the strongest region.
• Best month: 2024-05; weakest: 2024-12.
• 65 'Champion' customers (21% of revenue) — protect them with loyalty offers.
• Returns cost ₹31.5 lakh (4.1% of orders).
• Discounts cost ₹29.6 lakh without lifting units per order → review the discount policy.

RECOMMENDATIONS
1. Keep the top products in stock; bundle low-margin accessories with them.
2. Target 'At risk' customers with a win-back campaign.
3. Cap discounts at 10 % and test the effect (A/B test).
4. Investigate the causes of returns for the highest-value items.



---
## 🚀 Extensions
1. Replace the simulation with a real dataset from **Kaggle** (e.g., "Superstore Sales" or "Online Retail") and rerun every step.
2. Forecast next month's revenue with a moving average, then with `statsmodels`/`scikit-learn` linear regression.
3. Build a Streamlit dashboard with filters for region and date.
4. Add cohort analysis (customers grouped by first-purchase month → retention table).
5. Package the cleaning steps into a reusable function `clean_sales(df)` with unit tests.

**What you practised:** the full analyst workflow — messy data → clean → questions → numbers → recommendations.